# Glance Fashion Retrieval — End-to-End Pipeline
**Run on Kaggle with T4 GPU enabled.**  
Attach two input datasets before running:
- `fashionpedia-dataset` (uploaded dataset with 3,000 images + JSON)
- `glance-fashion-retrieval-code` (uploaded code zip)

> Run all cells top-to-bottom. The full pipeline takes approximately 30 minutes (indexing dominates).

---
## 0. Environment Setup

In [ ]:
import os, shutil, sys, warnings
warnings.filterwarnings('ignore')

# Auto-find and copy code from Kaggle input
code_source = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'src' in dirs and 'config.py' in os.listdir(os.path.join(root, 'src')):
        code_source = root
        break

if not code_source:
    raise FileNotFoundError('Could not find the codebase in /kaggle/input! Attach the code dataset.')

work_path = '/kaggle/working/glance-code'
if not os.path.exists(work_path):
    shutil.copytree(code_source, work_path)
    print('Copied code to: ' + work_path)
else:
    print('Code already at: ' + work_path)

if work_path not in sys.path:
    sys.path.insert(0, work_path)

print('Setup complete!')

In [ ]:
!pip install -q -r /kaggle/working/glance-code/requirements.txt
!pip install -q git+https://github.com/huggingface/transformers.git

---
## Step 1 — Subsample 3,000 Images

In [ ]:
from src.data_pipeline.subsample import subsample_images
print('Subsampling images...')
_ = subsample_images()

---
## Step 2 — Parse COCO Annotations into Structured Tags

In [ ]:
from src.data_pipeline.parse_annotations import parse_annotations
print('Parsing annotations...')
_ = parse_annotations()

---
## Step 3 — Generate Captions (No API Key Required)
> Uses structured Fashionpedia tags to derive captions and metadata locally.  
> Completes in approximately 2 minutes on CPU with zero network calls.

In [ ]:
from src.data_pipeline.generate_captions import generate_all_captions
print('Generating captions...')
generate_all_captions(resume=True)

---
## Step 4 — Build ChromaDB Vector Index
> Extracts FashionSigLIP embeddings, fuses image and text vectors,
> and upserts to ChromaDB. Takes approximately 20-25 minutes on T4 GPU.

In [ ]:
from src.indexer import build_index
print('Building vector index with Marqo-FashionSigLIP...')
build_index()

---
## Step 5 — Full Pipeline Evaluation
> Runs each eval query through query parsing, bi-encoder retrieval, and BLIP reranking.  
> Displays top-5 results with scores and a visual image grid.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

from src.retriever import search, search_without_reranker, _print_results
from src.config import EVAL_QUERIES
from src.utils import display_results, display_ablation_comparison, create_ablation_table
from IPython.display import display, Markdown

print('Running evaluation on', len(EVAL_QUERIES), 'queries...')

In [ ]:
full_results = {}
for q in EVAL_QUERIES:
    print('\n--- Query:', q, '---')
    results = search(q, top_n=5)
    full_results[q] = results
    _print_results(q, results)
    fig = display_results(q, results)
    display(fig)

---
## Step 6 — Ablation Study
> Compares retrieval quality with and without the BLIP reranker.
> Rendered as a Markdown table and a visual side-by-side image grid.

In [ ]:
baseline_results = {}
for q in EVAL_QUERIES:
    baseline_results[q] = search_without_reranker(q, top_n=5)

ablation_data = {
    'FashionSigLIP Only': baseline_results,
    'Full Pipeline (SigLIP + BLIP)': full_results,
}

table = create_ablation_table(EVAL_QUERIES, ablation_data)
display(Markdown(table))

In [ ]:
hard_query = EVAL_QUERIES[-1]
print('Visual comparison for:', hard_query)
fig = display_ablation_comparison(
    hard_query,
    baseline_results[hard_query],
    full_results[hard_query]
)
display(fig)